# Mycovirus discovery workflow orchestration notebook (Part 1)

Mycovirus discovery workflows accept your metratranscriptomic data and can be used for virus discovery with a particular focus on **mycoviruses** (including SRA mining workflows). 

The pipeline creates/uses a standardized project folder structure and a set of scripts to speed up analysis.

This notebook orchestrates the existing Slurm/HPC pipeline in this repository. It is intended to submit and monitor the wrapper scripts already provided in the folder: `Scripts/`, not replace them.


## What this notebook does
- verifies repository location and key files
- sets and validates `CONFIG`
- checks expected directories and adapters
- runs the wrapper scripts in order
- shows basic Slurm monitoring commands
- lists key output directories

## Important
- Run this notebook on powerPlant or a system with access to your Slurm cluster commands (`sbatch`, `squeue`, `sacct`).
- The pipeline scripts are expected to live in `Scripts/`.
- Review each submission cell before running it.


## 1. Edit your pipeline configuration
 
This pipeline expects a config file:
- `pipeline.env`

Edit the file saved in Scripts/, and set directories such as:

- `RAW_DIR`, `TRIM_DIR`, `MAPPING_DIR`, `CONTIGS_DIR`, `BLAST_DIR`, `LOG_DIR`, `ADAPTER_DIR`, etc.

to define where your project lives (so scripts don’t hardcode `/workspace/...` paths).

Before running any pipeline step, make sure **`PIPELINE_ROOT`** points to your actual workspace path.

Open the file using a text editor or throght the terminal. 

For example: 

```bash
nano ../Scripts/pipeline.env
```

## 2. Inspect configuration and important paths
This shell cell sources `config/pipeline.env` and reports key variables.

### Export `PIPELINE_ROOT` before running
Example (adjust to your environment):

A) REQUIRED: where the project folder lives (user chooses)

```bash
export USER="???"
export PIPELINE_ROOT="${PIPELINE_ROOT:-/workspace/$USER/Virus_discovery_workflows}"
```

## Configuration (`config/pipeline.env`)

The pipeline is designed to avoid hardcoded `/workspace/...` paths. Instead, you define your project layout in an environment file and export it when submitting jobs.

Typical variables include:

- `RAW_DIR` — raw FASTQs (e.g. `.../raw_reads`)
- `TRIM_DIR` — trimmed reads
- `MAPPING_DIR` — Bowtie2 non-host reads
- `CONTIGS_DIR` — assembly output (SPAdes contigs)
- `BLAST_DIR` — BLAST outputs
- `LOG_DIR` — Slurm stdout/stderr logs for wrappers/jobs
- `ADAPTER_DIR` — adapter FASTA location

Cluster-specific tools/DBs may also be configured via variables:
- `TRIMMOMATIC_JAR`
- `REF` (host reference fasta for Bowtie2 index)
Note: If you are running the example or your metatranscriptomic are from *B cinerea*. Here there is the genome of reference that was used. If you are using a different one please replace with its path. 

Databases: 

- `BLASTDB_NT`, `BLASTDB_RDRP`, `BLASTDB_RVDB`, etc.

## 3. Set up directories
Run this first to create the expected project layout.

Note: If you want to set a different project edit your pipeline.env and your 1_setup.sh files.

In [42]:
%%bash
set -euo pipefail
#set your project path
source "/workspace/hraczj/Mycovirus_discovery_workflows/Scripts/pipeline.env"
cd "/workspace/hraczj/Mycovirus_discovery_workflows/Scripts"
bash ./1_setup.sh

bash: fg: %%bash: no such job
Setting up project at: /workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline
Using PIPELINE_ROOT=/workspace/hraczj/Mycovirus_discovery_workflows
Using PROJECT=MVoP_pipeline
Setup complete.
Config installed at: /workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/config/pipeline.env
Next: run wrappers from /workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/scripts after editing PIPELINE_ROOT/PROJECT if needed.


## 4.  Make scripts executable (and fix CRLF if needed)
If you edited files on Windows and see `/bin/bash^M` errors, convert line endings to Unix LF:

In [57]:
#set your project path

source "/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/config/pipeline.env"
cd "/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/Scripts"

find -type f \( -name "*.sh" -o -name "*.slurm" \) -print0 | xargs -0 sed -i 's/\r$//'

chmod +x *.sh

ls

0_run_all.sh
10_pipeline_align_fasta.sh
10_pipeline_align_fasta.slurm
11_pipeline_iqtree_models.sh
11_pipeline_iqtree_models.slurm
1_pipeline_link_raw_fastqs_with_SSR_prefix.sh
1_setup.sh
2_pipeline_fastqc.sh
2_pipeline_fastqc.slurm
3_pipeline_trim.sh
3_pipeline_trim.slurm
4_pipeline_bowtie2_build_index.slurm
4_pipeline_bowtie2.sh
4_pipeline_bowtie2.slurm
5_pipeline_spades.sh
5_pipeline_spades.slurm
6_pipeline_blastn.sh
6_pipeline_blastn.slurm
6_pipeline_blastx_nr.slurm
6_pipeline_blastx.sh
7_pipeline_summary_result.sh
7_pipeline_summary_result.slurm
8_pipeline_extract_cotigs_by_id.sh
8_pipeline_genmark_diamond_from_nt_hits.sh
8_pipeline_genmark_diamond_from_nt_hits.slurm
8_pipeline_select_viral_contigs_from_hits.sh
9_pipeline_Extract_RdRp_Rep_AASequences_add_Reference_alignments.sh
NCBI_database_nr_update.slurm
NCBI_database_nt_update.slurm
pipeline_check_sra_downloads.sh
pipeline_download_sra.sh
pipeline_download_sra.slurm
pipeline_trim_assembly_abundance.sh
pipeline_trim_assembly_ab

## 5. Download SRA projects

In [58]:
source "/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/config/pipeline.env"
cd "/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/Scripts"
./pipeline_download_sra.sh

Submitting SRA download array: 3 accessions (0..2)
  ACCESSIONS_FILE=/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/accession_lists/accessions.txt
  OUTDIR=/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/results
Submitted job: 10249693
Monitor: sacct -j 10249693


## 6. Submit FastQC

This step submits the **FastQC** wrapper script to Slurm to perform initial quality control on sequencing reads (typically raw FASTQ files, and optionally trimmed reads).

### What FastQC does
- checks overall read quality across all base positions  
- reports GC content distribution and sequence length distribution  
- detects adapter contamination and overrepresented sequences  
- flags low-quality cycles (especially read ends)  
- summarizes pass/warn/fail metrics per sample

### Why this step is important
- provides an early snapshot of data quality before downstream analysis  
- helps decide whether trimming/filtering settings are appropriate  
- helps identify problematic samples that may affect assembly, mapping, and virus discovery results

### Expected outputs
- one `.html` report per FASTQ file (interactive summary)  
- one `.zip` result archive per FASTQ file (detailed module files)

Reports are written to your configured FastQC output directory (e.g., `FASTQC_DIR`) and job logs are written to `LOG_DIR`.

In [52]:
source "/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/config/pipeline.env"
cd "/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/Scripts"

./2_pipeline_fastqc.sh

Submitting FastQC jobs for files in: /workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/raw_reads
Submitted FastQC job: 10249654
Monitor with: sacct -j 10249654


## 7. Submit trimming

This step submits the trimming wrapper script to Slurm (e.g., Trimmomatic) to clean raw sequencing reads before host removal, assembly, and downstream viral discovery analyses.

### What trimming does
- removes adapter/primer contamination from reads  
- trims low-quality bases from read ends  
- applies sliding-window quality filtering to cut poor-quality regions  
- drops very short reads after trimming (minimum length filter)  
- keeps paired-end read synchronization (writes paired and unpaired outputs)

### Why this step is important
- improves overall read quality and downstream alignment/assembly performance  
- reduces false-positive hits caused by adapters or low-complexity/low-quality tails  
- increases confidence in contigs and BLAST/DIAMOND annotation results  
- helps standardize read quality across samples before comparative analysis

### Expected outputs

- **Output:** trimmed FASTQ files in `TRIM_DIR`, commonly:
  - `*_R1_paired.fastq.gz` / `*_R2_paired.fastq.gz` (used in most downstream steps)
  - `*_R1_unpaired.fastq.gz` / `*_R2_unpaired.fastq.gz` (optional, depending on workflow)

### What to check after completion
- number of reads retained vs removed (from trimmer logs)  
- whether adapter content and per-base quality improve in post-trim FastQC reports  
- whether paired output files are present for all samples

Logs are written to your configured `LOG_DIR`, and trimmed reads are written to `TRIM_DIR`.

In [1]:
source "/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/config/pipeline.env"
cd "/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/Scripts"

 ./3_pipeline_trim.sh

Submitting Trimmomatic job
CONFIG=/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/Scripts/../config/pipeline.env
RAW_DIR=/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/raw_reads
TRIM_OUT=/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/trimmed_reads
UNPAIRED=/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/trimmed_reads/unpaired
TRIM_LOG_DIR=/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/trimmed_reads/logs
CLIP=/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/adapters/Illumina.fa
TRIMMOMATIC_JAR=/software/bioinformatics/trimmomatic-0.39/trimmomatic-0.39.jar
Submitting Trimmomatic array: 3 samples (0..2)
Submitted: 10249724
Monitor:
  sacct -j 10249724
  squeue -u hraczj


## 8. Submit Bowtie2 host removal

This step submits Bowtie2 jobs to **remove host-derived reads** from your trimmed metatranscriptomic data.  
It runs after trimming and before assembly/viral annotation to enrich for potential viral (non-host) sequences.

### What this step does

The host-removal stage has two parts:

1. Build host index
   - uses your host reference genome FASTA (`REF`)
   - creates Bowtie2 index files under `INDEX_DIR` with basename `host`
   - this only needs to be repeated if the reference genome changes

2. Align trimmed reads and keep non-host pairs
   - finds paired trimmed files in `MAP_IN` matching patterns like:
   - maps each pair to the host index (`INDEX_DIR/host`)
   - discards mapped (host) reads
   - writes **unmapped paired reads** (candidate non-host reads) to `MAP_OUT` via `--un-conc-gz`

### Why this step is important

- reduces host background signal before assembly
- improves sensitivity for viral discovery by enriching non-host reads
- lowers computational load in downstream SPAdes/BLAST/DIAMOND steps
- helps reduce false-positive viral assignments caused by host-origin reads


### Expected outputs

- **Index files**: `${INDEX_DIR}/host.*.bt2` (or `.bt2l` depending on build)
- **Non-host paired reads** in `MAP_OUT` for each sample label:
  - `${LABEL}.non-host.fq.1.gz`
  - `${LABEL}.non-host.fq.2.gz`

### Notes and best practices

- ensure `REF` matches your host organism (replace if not *B. cinerea*)
- verify all input samples have both R1 and R2 files with consistent naming
- if no input files are found, confirm `MAP_IN` and trimming filename suffixes
- if mates are missing, check that trimming generated both paired files

This host-removal output is the input for the next assembly step (e.g., rnaviralSPAdes/SPAdes).


In [2]:
source "/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/config/pipeline.env"
cd "/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/Scripts"

./4_pipeline_bowtie2.sh

Submitting Bowtie2 index build job...
REF=/input/genomic/viral/Botrytis_cinerea_Refseq/Botrytiscinerea_RefSeq_B0510_ASM14353v4.fasta
INDEX_DIR=/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/annotation/bt2index
MAP_IN=/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/trimmed_reads
MAP_OUT=/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/mapping
MAP_LOG=/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/mapping/logs
Submitting Bowtie2 alignment job array (after index job 10249934)...
Submitted jobs:
  - index: 10249934
  - align: 10249935 (afterok:10249934)
Monitor:
  sacct -j 10249934,10249935
  squeue -u hraczj


## 9. Submit SPAdes / rnaviralspades

This step submits a Slurm array to run **rnaviralSPAdes** on Bowtie2-filtered non-host paired reads, performing **de novo assembly** of candidate viral transcripts/contigs for each sample.


### Why this step is important

- assembles short reads into longer contigs, improving detectability of viral sequences
- reconstructs candidate viral genome/transcript fragments from non-host reads
- provides contigs for downstream homology searches (BLAST/DIAMOND) and annotation
- enables sample-level comparison of recovered viral content


### Expected outputs

For each sample `${label}`:

- output folder: `${CONTIGS_DIR}/${label}/`
- main assembly files (typical SPAdes outputs), including:
  - `contigs.fasta`
  - `scaffolds.fasta` (if produced)
  - `transcripts.fasta` / graph-related files (tool-version dependent)

### Notes and best practices

- confirm Bowtie2 host-removal completed successfully before this step
- ensure every R1 has a matching R2 (`.fq.2.gz`)
- if no files are detected, verify `MAPPING_DIR` and filename pattern

In [3]:
source "/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/config/pipeline.env"
cd "/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/Scripts"


./5_pipeline_spades.sh

Submitting rnaviralSPAdes array for 3 samples (0..2)
SPADES_IN=/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/mapping
SPADES_OUT=/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/contigs
SPADES_LOG=/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/contigs/logs
Submitted: 10249984
Monitor:
  sacct -j 10249984
  squeue -u hraczj


## 10. Submit BLASTx and BLASTn searchers

This step submits similarity searches for assembled contigs to identify likely viral sequences and assign tentative taxonomy/function.

It runs three complementary searches:

1. **BLASTn (nucleotide vs nucleotide)** against `BLASTDB_NT`  
2. **BLASTx (translated nucleotide vs protein)** against `BLASTDB_NR`  
3. **DIAMOND blastx** against the curated `DIAMONDDB_RDRP` database 


### Why both BLASTn and BLASTx are used

- **BLASTn** is strong for close nucleotide-level matches (highly similar known viruses).
- **BLASTx** increases sensitivity for divergent viruses by searching at the protein level.
- **DIAMOND RdRp** provides a targeted, fast screen for RNA virus polymerase-like signals, useful for prioritizing candidate viral contigs.

Using all three improves discovery confidence and helps recover both known and divergent mycovirus-like sequences.


### Output format (tabular)

Search outputs are in tabular format (`outfmt 6`) and include key fields such as:
- query contig ID
- subject accession/ID
- percent identity
- alignment length
- e-value
- bit score
- taxonomy fields (where available, e.g., `staxid`, `ssciname`)

These tables are used downstream for filtering, candidate ranking, and summary reporting.


In [58]:
source "/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/config/pipeline.env"
cd "/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/Scripts"

./6_pipeline_blastx.sh
./6_pipeline_blastn.sh

Resolved BLASTDB_NR:
  input:    /input/genomic/viral/DBs/NCBI_NonHumanViral_nr_May2025/NCBI_NonHuman_ViralSequences_nr_May2025
  resolved: /input/genomic/viral/DBs/NCBI_NonHumanViral_nr_May2025/NCBI_NonHuman_ViralSequences_nr_May2025
Submitting blastx (nr) array for 3 samples (0..2)
  DB=/input/genomic/viral/DBs/NCBI_NonHumanViral_nr_May2025/NCBI_NonHuman_ViralSequences_nr_May2025
Submitting diamond (rdrp) array for 3 samples (0..2)
  DB=/input/genomic/viral/DBs/RdRp-scan/RdRp-scan_0.90.dmnd
Submitted jobs:
  - blastx_nr:    10250870
  - diamond_rdrp: 10250871
Monitor:
  sacct -j 10250870,10250871
  squeue -u hraczj
Track jobs individually:
  sacct -j 10250870
  sacct -j 10250871
Submitting BLASTn array for 3 samples (0..2)
IN=/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/contigs
OUT=/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/blast_results
BLASTDB_NT=/input/genomic/viral/DBs/NCBI_NonHumanViral_nt_May2025
Submitted jobs:
  - blastn: 10250872
Monitor:


## 12. Submit summary step


In [41]:
source "/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/config/pipeline.env"
cd "/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/Scripts"

./7_pipeline_summary_result.sh

Submitting R_summary_result job
Submitted jobs:
  - R_summary_result: 10250703
Monitor:
  sacct -j 10250703
  squeue -u hraczj


## 13. Monitor Slurm jobs
Run these cells after submission to inspect queued/running/completed jobs.

In [25]:
%%bash
set -euo pipefail
echo "USER=${USER:-$(hraczj)}"
squeue -u "${USER:-$(hraczj)}" || true

bash: fg: %%bash: no such job
USER=hraczj
             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)
          10250525      fast gm_dia_S   hraczj PD       0:00      1 (Nodes required for job are DOWN, DRAINED or reserved for jobs in higher priority partitions)
        10250139_0      fast  searchx   hraczj  R    1:39:51      1 aklppb43
        10250139_1      fast  searchx   hraczj  R    1:39:51      1 aklppb44
        10250139_2      fast  searchx   hraczj  R    1:39:51      1 aklppb30
    10249984_[0-2]     short rnaviral   hraczj PD       0:00      1 (Priority)


In [51]:
set -euo pipefail
echo 'Recent accounting entries:'
sacct -u "${USER:-$(hraczj)}" --format=JobID,JobName%30,State,Elapsed,MaxRSS,ExitCode | tail -n 30 || true

Recent accounting entries:
10250703                   R_summary_result  COMPLETED   00:04:57                 0:0 
10250703.ba+                          batch  COMPLETED   00:04:57   8387924K      0:0 
10250703.ex+                         extern  COMPLETED   00:04:57                 0:0 
10250797                 gm_dia_SRR11783490     FAILED   00:00:04                 1:0 
10250797.ba+                          batch     FAILED   00:00:04     19644K      1:0 
10250797.ex+                         extern  COMPLETED   00:00:04                 0:0 
10250809                 gm_dia_SRR11783490     FAILED   00:00:04                 1:0 
10250809.ba+                          batch     FAILED   00:00:04     27016K      1:0 
10250809.ex+                         extern  COMPLETED   00:00:04                 0:0 
10250815                 gm_dia_SRR11783490     FAILED   00:00:03                25:0 
10250815.ba+                          batch     FAILED   00:00:03     18488K     25:0 
10250815.ex+    

# Mycovirus discovery workflow orchestration notebook (Part 2)

## 14. ORF prediction + protein screening against RdRp (ORF finder / GeneMark + DIAMOND)

This step starts from candidate viral contigs (selected from filtered NT hits), chooses a prediction strategy based on expected genome type (RNA-like vs DNA-like), generates translated proteins, and screens them against an RdRp reference database using DIAMOND.

### What this step does

1. Select candidate contigs from NT-hit table
2. Extract selected contig sequences
3. Choose genome-type branch (RNA-like vs DNA-like)
   - Controlled by:
     - `VIRUS_GENOME_HINT=auto|rna|dna` (default `auto`)
   - In `auto` mode, branch is inferred from max selected contig length:
     - if `max_len >= DNA_INTRON_AWARE_MIN_BP` (default 20000) → **DNA-like / intron-aware branch**
     - otherwise → **RNA-like / intronless ORF finder branch**
   - Logged output includes selected branch:
     - `branch=rna_orf` or `branch=dna_gmhmme3`

4. RNA-like branch (`rna_orf`)
   - Uses an internal intronless 6-frame ORF finder (no GeneMark dependency for translation).
  
5. DNA-like branch (`dna_gmhmme3`)
   - Runs `gmhmme3` with an auto-detected model from `heu_dir/*.mod`
   - Internal parser reconstructs CDS/proteins from `genemark.pred` exon coordinates (cluster-portable fallback when GeneMark helper utilities differ)
6. Annotate predicted proteins with DIAMOND (`blastp`)

### Important behavior

- **Zero DIAMOND hits are valid outcomes** (not treated as pipeline failure).
- If no matches are found, `${SAMPLE}.rdrp_diamond.tsv` is left empty and a warning is logged.
- This indicates “no RdRp-like hit under current thresholds/database,” not a software crash.

### Why this step is important

- Moves from nucleotide-level evidence to **protein-level evidence**
- Uses a genome-architecture-aware prediction strategy:
  - intronless ORF mode for RNA-like compact genomes
  - intron-aware GeneMark mode for DNA-like candidates when appropriate
- Detects hallmark viral proteins (especially **RdRp-like proteins**)
- Prioritizes contigs for downstream curation and phylogeny

### Typical run command

```bash
bash 8_pipeline_genmark_diamond_from_nt_hits.sh <SAMPLE_ID> <nt_filtered_hits.tsv>
```

In [52]:
source "/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/config/pipeline.env"
cd "/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/Scripts"

bash 8_pipeline_genmark_diamond_from_nt_hits.sh SRR11783490 /workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/blast_results/nt/nt_filtered_hits.tsv

Submitted: 10250868 sample=SRR11783490 out=/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/annotation/orf_and_phylo/SRR11783490
10250868


## 15. Assign one viral family per contig (qseqid) from your NT hits table.

### What it does

- Reads nt_filtered_hits.tsv
- Groups rows by qseqid
- Picks the top bitscore hit per qseqid
- Uses the Family value from that top hit
- If top-hit family is missing (NA/blank), labels as Unclassified
- Writes a 2-column table

In [57]:
source "/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/config/pipeline.env"
cd "/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/Scripts"

python3 assign_family.py /workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/annotation/orf_and_phylo/SRR11783490/SRR11783490.nt_filtered_hits.tsv /workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/annotation/orf_and_phylo/SRR11783490/SRR11783490.assigned_family.tsv

Saved: /workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/annotation/orf_and_phylo/SRR11783490/SRR11783490.assigned_family.tsv


Notes
- Review job outputs in `$LOG_DIR` and any per-step log directories.
- For large runs, submit one step and confirm outputs before proceeding to the next.

## 16. Extract RdRp/Rep Amino Acid Sequences and add to Reference Mycoviral families alignments

1.	Located the reference alignment file in FASTA format from the ICTV report page for the family Botourmiaviridae and saved it as Botourmiaviridae_RdRp_family.fasta at (https://ictv.global/sites/default/files/report_files/OPSR.Botourma.Fig4_.v11.trim_alignment_renamed.txt)
  
2.	Download the FASTA file from the command line using curl:

In [2]:
curl -L -A "Mozilla/5.0" \
  -o /workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/annotation/orf_and_phylo/SRR11783490/Botourmiaviridae_RdRp_family.fasta \
 "https://ictv.global/sites/default/files/report_files/OPSR.Botourma.Fig4_.v11.trim_alignment_renamed.txt"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 87955    0 87955    0     0  59711      0 --:--:--  0:00:01 --:--:-- 59711


3. Run combine_fasta.pyPython script to merge the viral read with the reference alignment. The script should take two FASTA files as input (the reference alignment and the file containing the viral read) and generate a combined output FASTA file.

In [2]:
source "/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/config/pipeline.env"
cd "/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/Scripts"

chmod +x combine_fasta.py
./combine_fasta.py  \
/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/annotation/orf_and_phylo/SRR11783490/viral_read.fasta  \
/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/annotation/orf_and_phylo/SRR11783490/Botourmiaviridae_RdRp_family.fasta

Combined FASTA written to /workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/annotation/orf_and_phylo/SRR11783490/Botourmiaviridae_RdRp_family_ViralReadasignment.fasta


## 17.	Perform a multiple sequence alignment of the RdRp amino acid sequences.
  

In [3]:
cd "/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/annotation/orf_and_phylo/SRR11783490"

module load mafft

mafft --auto Botourmiaviridae_RdRp_family_ViralReadasignment.fasta > Botourmiaviridae_RdRp_family_ViralReadasignment_aligned.fasta



All-to-all alignment.
tbfast-pair (aa) Version 7.307 alg=L, model=BLOSUM62, 2.00, -0.10, +0.10, noshift, amax=0.0
0 thread(s)

Loading 'hat3.seed' ... 
done.
Writing hat3 for iterative refinement
Gap Penalty = -1.53, +0.00, +0.00
treein = 0
compacttree = 0
Constructing a UPGMA tree ... 
  170 / 172
done.

Progressive alignment ... 
STEP   171 /171 c
done.
tbfast (aa) Version 7.307 alg=A, model=BLOSUM62, 1.53, -0.00, -0.00, noshift, amax=0.0
0 thread(s)

minimumweight = 0.000010
autosubalignment = 0.000000
nthread = 0
randomseed = 0
blosum 62 / kimura 200
poffset = 0
niter = 2
sueff_global = 0.100000
Loading 'hat3' ... done.

  170 / 172
Segment   1/  1    1- 796
done 002-001-1  identical.   
dvtditr (aa) Version 7.307 alg=A, model=BLOSUM62, 1.53, -0.00, -0.00, noshift, amax=0.0
0 thread(s)


Strategy:
 L-INS-i (Probably most accurate, very slow)
 Iterative refinement method (<2) with LOCAL pairwise alignment information

If unsure which option to use, try 'mafft --auto input > output'

## 18.	Model Selection and Construct the phylogenetic tree using IQ-TREE with default settings and 1,000 ultrafast bootstrap replicates.

Identify the headers corresponding to Narnavirus so they can be assigned as the outgroup.

Run the following command in the Linux console:


In [4]:
cd "/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/annotation/orf_and_phylo/SRR11783490"

grep '^>' Botourmiaviridae_RdRp_family_ViralReadasignment_aligned.fasta  | grep -i 'narnavirus'

>'AF039063_NOT_ON_SPREADSHEET_Saccharomyces_20S_RNA_narnavirus_.1'
>'U90136_NOT_ON_SPREADSHEET_Saccharomyces_23S_RNA_narnavirus_.1'


Load IQ-TREE and run the tree inference with:

In [11]:
cd "/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/annotation/orf_and_phylo/SRR11783490"

module load iqtree

iqtree -s Botourmiaviridae_RdRp_family_ViralReadasignment_aligned.fasta -st AA -m MFP -B 1000

IQ-TREE multicore version 2.1.2 COVID-edition for Linux 64-bit built Mar 30 2021
Developed by Bui Quang Minh, James Barbetti, Nguyen Lam Tung,
Olga Chernomor, Heiko Schmidt, Dominik Schrempf, Michael Woodhams.

Host:    aklppj31.pfr.co.nz (AVX512, FMA3, 93 GB RAM)
Command: iqtree -s Botourmiaviridae_RdRp_family_ViralReadasignment_aligned.fasta -st AA -m MFP -B 1000
Seed:    118432 (Using SPRNG - Scalable Parallel Random Number Generator)
Time:    Tue Jun 30 09:42:59 2026
Kernel:  AVX+FMA - 1 threads (10 CPU cores detected)

HINT: Use -nt option to specify number of threads because your CPU has 10 cores!
HINT: -nt AUTO will automatically determine the best number of threads to use.

Reading alignment file Botourmiaviridae_RdRp_family_ViralReadasignment_aligned.fasta ... Fasta format detected
Alignment most likely contains protein sequences
Alignment has 172 sequences with 772 columns, 485 distinct patterns
402 parsimony-informative, 39 singleton sites, 331 constant sites
'MK584845_Acrem

Run tree_genus_report.py, a python script to Identify the genus of each sequence, listed the assigned sequences under each genus in text format, and reported the clade in which Viral_read_1 is grouped.

In [13]:
source "/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/config/pipeline.env"
cd "/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/Scripts"

chmod +x tree_genus_report.py
./tree_genus_report.py \
/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/annotation/orf_and_phylo/SRR11783490/Botourmiaviridae_RdRp_family_ViralReadasignment_aligned.fasta.contree

Table written to /workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/annotation/orf_and_phylo/SRR11783490/Botourmiaviridae_RdRp_family_ViralReadasignment_aligned.fasta_genus_table.txt

Clade containing Viral_read_1:
(Viral_read_1:0.02295,_MK584845_Acremonium_sclerotigenum_ourmia-like_virus_1_:0.00141)100.00:0.34556;

Inferred genus category for the Viral_read_1 clade: Acremonium


Plot the tree and mark the nodes where this support was >70%, the output of this should be figure (PNG) format.
Using a python script called plot_bootstrap_tree.py

In [2]:
source "/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/config/pipeline.env"
cd "/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/Scripts"

chmod +x plot_bootstrap_tree.py
./plot_bootstrap_tree.py \
/workspace/hraczj/Mycovirus_discovery_workflows/MVoP_pipeline/annotation/orf_and_phylo/SRR11783490/Botourmiaviridae_RdRp_family_ViralReadasignment_aligned.fasta.contree \
--width 30 \
  --height 45 \
  --dpi 300 \
  --label-size 10 \
  --bootstrap-size 10 \
  --marker-size 2.5 \
  --no-show

Saved: Botourmiaviridae_RdRp_family_ViralReadasignment_aligned.fasta.contree.bootstrap_gt70.png
Viral_read_* labels colored red: 1
Bootstrap nodes highlighted: 159


Note: If Biopython is missing: python3 -m pip install biopython matplotlib